In [1]:
from pathlib import Path

import pandas as pd

In [2]:
raw_dir = Path("../data/raw")
csv_files = sorted(raw_dir.glob("sim_events_*.csv"))
csv_files

[WindowsPath('../data/raw/sim_events_20251117_163516.csv'),
 WindowsPath('../data/raw/sim_events_20251117_170014.csv'),
 WindowsPath('../data/raw/sim_events_20251117_170029.csv'),
 WindowsPath('../data/raw/sim_events_20251117_170042.csv')]

In [3]:
all_dfs = []

for csv_path in csv_files:
    # run_id ricavato dal nome del file, es. "sim_events_20250101_153000"
    run_id = csv_path.stem  # nome file senza estensione

    df = pd.read_csv(csv_path)
    df["run_id"] = run_id

    all_dfs.append(df)

len(all_dfs)

4

In [4]:
df_all = pd.concat(all_dfs, ignore_index=True)
df_all.head()

,event_type,convoy_id,minute,step_index,progress,run_id
0,convoy_spawned,1,0,-1,0.0,sim_events_20251117_163516
1,convoy_spawned,2,0,-1,0.0,sim_events_20251117_163516
2,convoy_spawned,3,0,-1,0.0,sim_events_20251117_163516
3,convoy_spawned,4,0,-1,0.0,sim_events_20251117_163516
4,convoy_spawned,5,0,-1,0.0,sim_events_20251117_163516


In [5]:
df_all.columns.tolist()

['event_type', 'convoy_id', 'minute', 'step_index', 'progress', 'run_id']

In [6]:
run_meta = (
    df_all
    .groupby("run_id")["minute"]
    .max()
    .reset_index()
    .rename(columns={"minute": "total_minutes"})
)

run_meta

,run_id,total_minutes
0,sim_events_20251117_163516,120
1,sim_events_20251117_170014,0
2,sim_events_20251117_170029,30
3,sim_events_20251117_170042,500


In [7]:
def estimate_timestep_minutes(df_run: pd.DataFrame) -> int:
    unique_minutes = sorted(df_run["minute"].unique())
    if len(unique_minutes) < 2:
        return unique_minutes[0] if unique_minutes else 0
    # differenza tra i primi due valori distinti
    return int(unique_minutes[1] - unique_minutes[0])


timestep_list = []

for run_id, df_run in df_all.groupby("run_id"):
    ts = estimate_timestep_minutes(df_run)
    timestep_list.append({"run_id": run_id, "timestep_minutes": ts})

run_timestep = pd.DataFrame(timestep_list)
run_timestep

,run_id,timestep_minutes
0,sim_events_20251117_163516,10
1,sim_events_20251117_170014,0
2,sim_events_20251117_170029,30
3,sim_events_20251117_170042,500


In [8]:
run_info = pd.merge(run_meta, run_timestep, on="run_id", how="left")
run_info

,run_id,total_minutes,timestep_minutes
0,sim_events_20251117_163516,120,10
1,sim_events_20251117_170014,0,0
2,sim_events_20251117_170029,30,30
3,sim_events_20251117_170042,500,500


In [9]:
rows = []

grouped = df_all.groupby(["run_id", "convoy_id"])

for (run_id, convoy_id), group in grouped:
    # meta info per questa run
    info = run_info[run_info["run_id"] == run_id].iloc[0]
    total_minutes = int(info["total_minutes"])
    timestep_minutes = int(info["timestep_minutes"])

    # convoglio completato?
    df_completed = group[group["event_type"] == "convoy_completed"]
    is_completed = len(df_completed) > 0

    # per ora non usiamo il minuto di completamento come feature,
    # ma potresti aggiungerlo se ti serve
    if is_completed:
        completion_minute = int(df_completed["minute"].min())
    else:
        completion_minute = None

    # Definizione semplice di rischio:
    # - 0 = convoglio completato
    # - 1 = convoglio non completato
    risk_label = 0 if is_completed else 1

    row = {
        "run_id": run_id,
        "convoy_id": int(convoy_id),
        "total_minutes": total_minutes,
        "timestep_minutes": timestep_minutes,
        "is_completed": is_completed,
        "completion_minute": completion_minute,
        "risk_label": risk_label,
    }

    rows.append(row)

df_convoys = pd.DataFrame(rows)
df_convoys.head()

,run_id,convoy_id,total_minutes,timestep_minutes,is_completed,completion_minute,risk_label
0,sim_events_20251117_163516,1,120,10,True,10.0,0
1,sim_events_20251117_163516,2,120,10,True,10.0,0
2,sim_events_20251117_163516,3,120,10,True,10.0,0
3,sim_events_20251117_163516,4,120,10,True,10.0,0
4,sim_events_20251117_163516,5,120,10,True,10.0,0


In [10]:
len(df_convoys)

18

In [11]:
df_convoys["risk_label"].value_counts()

risk_label
1    10
0     8
Name: count, dtype: int64

In [12]:
df_convoys["is_completed"].value_counts()

is_completed
False    10
True      8
Name: count, dtype: int64

In [13]:
df_convoys[df_convoys["risk_label"] == 1]["run_id"].value_counts()

run_id
sim_events_20251117_170014    10
Name: count, dtype: int64

In [14]:
output_path = "../data/processed/convoy_risk_dataset.csv"
df_convoys.to_csv(output_path, index=False)
output_path

'../data/processed/convoy_risk_dataset.csv'

In [15]:
print("Righe dataset:", len(df_convoys))
print("Distribuzione risk_label:")
print(df_convoys["risk_label"].value_counts())

Righe dataset: 18
Distribuzione risk_label:
risk_label
1    10
0     8
Name: count, dtype: int64
